# Assignment #4: Soft Computer, Reworked with Hand-Tagged Semantics

reworking my first assignment (the soft computer poetry generator) to use our class's hand-tagged semantic corpus. instead of picking from small hardcoded vocabulary lists i wrote myself, each line now pulls from the nearest neighbors of a target vector in the shared tag-space. the structural scaffolding of the original poem stays, but the vocabulary comes from the class's collective sense of which words feel organic, warm, sharp, emotive, etc.

In [38]:
import json
import random
from collections import defaultdict, Counter
from simpleneighbors import SimpleNeighbors

In [39]:
%pip install simpleneighbors scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.


In [40]:
# loading the tagged data
lines = open("semantic-categories-2026-04-10.jsonl").readlines()

cats = defaultdict(list)
for item in lines:
    obj = json.loads(item)
    if len(obj['label']) > 0:
        cats[obj['text'].lower()].extend(obj['label'])

In [41]:
# count tags per word
cats_counted = {word: Counter(tags) for word, tags in cats.items()}

In [42]:
# build the column order for vectors
unique_features = set()
for val in cats.values():
    unique_features.update(val)

cols = list(unique_features)
name2idx = {name: i for i, name in enumerate(cols)}
print(cols)

['cold', 'sharp', 'warm', 'tangible', 'organic', 'round', 'emotive', 'academic', 'outlandish', 'youthful']


In [43]:
# vectorize every word
cats_vectorized = {}
for word, counts in cats_counted.items():
    cats_vectorized[word] = [counts[feat] for feat in cols]

In [44]:
# build the nearest-neighbor lookup
lookup = SimpleNeighbors(len(cols), metric="angular")
for word, vec in cats_vectorized.items():
    lookup.add_one(word, vec)
lookup.build()

In [45]:
# helper to build target vectors by tag name
def make_vector(**kwargs):
    vec = [0] * len(cols)
    for k, v in kwargs.items():
        vec[name2idx[k]] = v
    return vec

In [61]:
# generate poem
stanza_count = 7

for i in range(stanza_count):
    material   = random.choice(lookup.nearest(make_vector(organic=2, tangible=2), n=20))
    structure  = random.choice(lookup.nearest(make_vector(round=2, warm=1), n=20))
    power      = random.choice(lookup.nearest(make_vector(warm=3, organic=1), n=20))
    interface  = random.choice(lookup.nearest(make_vector(tangible=2, sharp=1), n=20))
    location   = random.choice(lookup.nearest(make_vector(warm=2, round=2), n=20))
    inhabitant = random.choice(lookup.nearest(make_vector(emotive=3, warm=1), n=20))
    promise    = random.choice(lookup.nearest(make_vector(emotive=2, youthful=1), n=20))

    print()
    print(f"a computer of {material}")
    print(f"     bound by {structure}")
    print(f"          powered by {power}")
    print(f"                with {interface}")
    print(f"                     resting among {location}")
    print(f"                          inhabited by {inhabitant}")
    print(f"                               and it offers {promise}.")


a computer of threads
     bound by okay
          powered by simmering
                with hunger
                     resting among lovely
                          inhabited by confused
                               and it offers slang.

a computer of men singers
     bound by message
          powered by praise
                with vehicle
                     resting among everyone
                          inhabited by dad
                               and it offers fame.

a computer of tall
     bound by lyric
          powered by red
                with gunpowder
                     resting among nod
                          inhabited by personal
                               and it offers fame.

a computer of period
     bound by lyric
          powered by sunset
                with fit
                     resting among okay
                          inhabited by concerns
                               and it offers jolly.

a computer of chestnut
     bound by salon


## rejection

In [47]:
# helper to check if a word has forbidden tags
def has_forbidden(word, forbidden):
    """return True if the word was tagged with any forbidden category"""
    tags = set(cats_counted.get(word, Counter()).keys())
    return bool(tags & forbidden)

def pick_soft(target_vec, forbidden, pool_size=40, max_tries=20):
    """pick a word near target_vec that has none of the forbidden tags"""
    candidates = lookup.nearest(target_vec, n=pool_size)
    for _ in range(max_tries):
        word = random.choice(candidates)
        if not has_forbidden(word, forbidden):
            return word
    # if can't find a clean one, return the last candidate anyway
    return word

In [65]:
# generate, refusing cold/sharp/academic words
forbidden = {"cold", "sharp", "academic"}

stanza_count = 7

for i in range(stanza_count):
    material   = pick_soft(make_vector(organic=2, tangible=2), forbidden)
    structure  = pick_soft(make_vector(round=2, warm=1), forbidden)
    power      = pick_soft(make_vector(warm=3, organic=1), forbidden)
    interface  = pick_soft(make_vector(tangible=2), forbidden)
    location   = pick_soft(make_vector(warm=2, round=2), forbidden)
    inhabitant = pick_soft(make_vector(emotive=3, warm=1), forbidden)
    promise    = pick_soft(make_vector(emotive=2, youthful=1), forbidden)

    print()
    print(f"a computer of {material}")
    print(f"     bound by {structure}")
    print(f"          powered by {power}")
    print(f"                with {interface}")
    print(f"                     resting among {location}")
    print(f"                          inhabited by {inhabitant}")
    print(f"                               and it offers {promise}.")


a computer of bodies
     bound by salon
          powered by share
                with ground
                     resting among belly
                          inhabited by daily
                               and it offers _statirical_.

a computer of daughter
     bound by character
          powered by parent
                with ground
                     resting among hearts
                          inhabited by personal
                               and it offers jolly.

a computer of gasoline
     bound by message
          powered by father
                with trip
                     resting among lamp
                          inhabited by chuckle
                               and it offers sentimental.

a computer of threads
     bound by open
          powered by heaven
                with side
                     resting among the sun
                          inhabited by daily
                               and it offers influences.

a computer of sand
     b

## tag-descriptor

In [49]:
# helper to read top tags for a word
def top_tags(word, n=2):
    """return the n most-tagged categories for a word, as a list"""
    counts = cats_counted.get(word, Counter())
    return [tag for tag, _ in counts.most_common(n)]

In [71]:
# generate with tag descriptors
stanza_count = 7

for i in range(stanza_count):
    material   = random.choice(lookup.nearest(make_vector(organic=2, tangible=2), n=20))
    structure  = random.choice(lookup.nearest(make_vector(round=2, warm=1), n=20))
    power      = random.choice(lookup.nearest(make_vector(warm=3, organic=1), n=20))
    interface  = random.choice(lookup.nearest(make_vector(tangible=2, sharp=1), n=20))
    location   = random.choice(lookup.nearest(make_vector(warm=2, round=2), n=20))
    inhabitant = random.choice(lookup.nearest(make_vector(emotive=3, warm=1), n=20))
    promise    = random.choice(lookup.nearest(make_vector(emotive=2, youthful=1), n=20))

    def describe(word):
        tags = top_tags(word, 2)
        return f"{word} ({', '.join(tags)})" if tags else word

    print()
    print(f"a computer of {describe(material)}")
    print(f"     bound by {describe(structure)}")
    print(f"          powered by {describe(power)}")
    print(f"                with {describe(interface)}")
    print(f"                     resting among {describe(location)}")
    print(f"                          inhabited by {describe(inhabitant)}")
    print(f"                               and it offers {describe(promise)}.")


a computer of tall (organic, tangible)
     bound by ease (round, warm)
          powered by afternoon (warm, organic)
                with stitch (tangible, sharp)
                     resting among breast (round, warm)
                          inhabited by dread (emotive)
                               and it offers excitement (emotive, youthful).

a computer of woman (organic, tangible)
     bound by mutual (round)
          powered by simmering (warm, organic)
                with dock (tangible, cold)
                     resting among lamp (round, warm)
                          inhabited by concerns (emotive, warm)
                               and it offers performances (emotive, youthful).

a computer of woman (organic, tangible)
     bound by harmonious (round, warm)
          powered by red (warm, organic)
                with vehicle (tangible, sharp)
                     resting among unusually good (emotive, round)
                          inhabited by melancholies (e